# Project 11 · Deep Research Analyst (Reference solution)

**Module 11 · Deep Agents** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 6–8 hours |
| 📋 Prerequisites | Lessons 11.01–11.11; Module 06 (RAG and citations) |
| 🔑 Keys | none required · `GOOGLE_API_KEY` (or your `MASTERY_DEEP_MODEL` key) for the optional LIVE run; `GROQ_API_KEY` or `TAVILY_API_KEY` for live web search |
| 💸 Cost (LIVE) | smoke set (3 questions) ≈ 45–75K tokens; full set (20) ≈ 300–500K tokens: spread it over several days on a free tier |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |
| 📄 Spec | [project/README.md](../README.md) |

**The job:** Orbita Labs' sales and support teams get questions that need a short, *sourced* answer: "how often must a Marine station be rinsed?", "what is the API rate limit?", and some that the documents can't answer. You build the analyst that researches the documents (and the web if needed), has its findings checked by a critic, and writes a brief a human approves before it is saved. Then you prove it works with an evaluation and write the report your lead reads.

This is the **reference solution**. Try the [starter notebook](../11-project-deep-research-analyst.ipynb) first; compare afterwards (same cells, same order).

## Requirements (from the spec)

1. **Agent:** a coordinator with a `researcher` (only `search_docs` and `web_search`) and a `critic` (no tools); system prompts that define the workflow, the brief format and the citation style.
2. **Safety:** secret files denied; **every** write under `/reports/` paused for review; ≤ 12 model calls per run.
3. **Review policy:** a function that approves good briefs and rejects, with reasons, briefs that miss a section, answer without sources, or decline with high confidence.
4. **Evaluation:** a checker for format, grounding (against what the tools returned), workflow, budget and correctness, validated on fixtures; run it on all 20 questions offline (and the smoke set live).
5. **Report:** a Markdown report with a SHIP/HOLD decision that follows the bar, generated from the measured numbers.

**Modes:** everything runs **offline** with scripted models that follow the workflow (`offline_models()`), so Restart & Run All always completes for free. With a key, Step 7 runs the same analyst on your real models.

| TODO | Where | What |
|---|---|---|
| 1 | Step 2 | `PROMPTS` for coordinator, researcher and critic |
| 2 | Step 3 | `build_analyst(models, tools, checkpointer=...)` |
| 3 | Step 4 | `review(action)`: the human-review policy |
| 4 | Step 5 | `check_brief(run, q)` |
| 5 | Step 8 | `write_report(path)` |

---
## Step 0 · Setup

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/11-deep-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GOOGLE_API_KEY', 'GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm11-project'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.deep import deep_key_ready, deep_model_spec, fallback_specs

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")

DEEP_MODEL = deep_model_spec("main")              # MASTERY_DEEP_MODEL (course default: Gemini free tier)
SUBAGENT_MODEL = deep_model_spec("subagent")      # MASTERY_DEEP_SUBAGENT_MODEL
HAVE_DEEP_KEY = deep_key_ready(DEEP_MODEL)        # the key for THAT provider (GOOGLE_API_KEY for google_genai:)
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m11-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("deep model:", DEEP_MODEL, "| subagents:", SUBAGENT_MODEL)
print("fallbacks :", fallback_specs(DEEP_MODEL) or "none (optional: MASTERY_FALLBACK_MODELS in .env)")
if not HAVE_DEEP_KEY:
    print(f"\n⚠️ No key for {DEEP_MODEL.split(':')[0]}: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Step 1 · The documents, the questions and the tools

The Orbita Labs documents from Module 06 (Markdown and PDF), 20 golden questions with the expected source, and two research tools that **log every source they return** in a ledger, so you can check a brief's citations even when a subagent made the call.

In [ ]:
# The building blocks (src/mastery/deep/research.py): tools that log their sources, questions, offline models.
import json
import re
from pathlib import Path

from deepagents import FilesystemPermission
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from mastery.config import find_repo_root
from mastery.deep import build_deep_agent, file_text, run_stats, secret_deny_rules, trajectory
from mastery.deep.evals import subagents_called
from mastery.deep.research import (BRIEF_SECTIONS, SourceLedger, brief_markdown, cited_docs, load_questions,
                                   offline_models, research_tools, section)
from mastery.evals.stats import wilson_interval

OUT_DIR = find_repo_root() / "data" / "outputs" / "m11"
OUT_DIR.mkdir(parents=True, exist_ok=True)

QUESTIONS = load_questions()                 # 20: 16 answerable from the Orbita documents, 4 to decline
SMOKE = load_questions(smoke=True)           # 3: a Markdown fact, a PDF fact, a question to decline
ledger = SourceLedger()
search_docs, web_search = research_tools(ledger)
print(len(QUESTIONS), "questions;", sum(q["in_corpus"] for q in QUESTIONS), "answerable. Smoke:", [q["id"] for q in SMOKE])
print("example:", QUESTIONS[0])

**You should now see** `20 questions; 16 answerable. Smoke: ['g01', 'g06', 'g17']` and the first question.

---
## Step 2 · The prompts

In [ ]:
PROMPTS = {
    "coordinator": (
        "You are a research analyst. For every question: 1) delegate the research to the `researcher` subagent; "
        "2) send its findings to the `critic` subagent; 3) write ONE brief to /reports/<short-slug>.md with the "
        "sections ## Answer, ## Evidence, ## Sources, ## Confidence. Cite documents as [doc: path] and web pages by URL, "
        "only sources the researcher returned. If nothing reliable was found, say so in ## Answer and set confidence to low."),
    "researcher": (
        "Search the Orbita documents with search_docs first; use web_search only if the documents don't answer. "
        "Return JSON: {\"answer\": ..., \"citations\": [{\"source\": path or URL, \"quote\": exact words}]}. "
        "Quote word for word. If nothing answers the question, return an empty citations list."),
    "critic": (
        "You check research findings. Reply 'OK: ...' if every claim has a quoted source that supports it, "
        "otherwise 'REJECT: ...' listing each unsupported claim. Never add facts."),
}

In [ ]:
def prompt_problems(prompts):
    c, r, k = prompts["coordinator"], prompts["researcher"], prompts["critic"]
    needs = {"coordinator": ["researcher", "critic", "/reports/", "[doc:", *BRIEF_SECTIONS], "researcher": ["search_docs", "web_search"],
             "critic": ["OK", "REJECT"]}
    return {name: [w for w in words if w.lower() not in prompts[name].lower()] for name, words in needs.items()
            if any(w.lower() not in prompts[name].lower() for w in words)}


print("prompt problems:", prompt_problems(PROMPTS) or "none")

**You should now see** `prompt problems: none` (solution). Starter: every missing word, by prompt.

---
## Step 3 · The analyst

In [ ]:
def build_analyst(models, tools, *, checkpointer):
    search_docs, web_search = tools
    researcher = {"name": "researcher", "model": models["researcher"], "tools": [search_docs, web_search],
                  "description": "Finds evidence in the Orbita documents (and the web if needed) and returns quoted findings as JSON.",
                  "system_prompt": PROMPTS["researcher"]}
    critic = {"name": "critic", "model": models["critic"], "tools": [],
              "description": "Checks findings: every claim needs a quoted source. Replies OK or REJECT.",
              "system_prompt": PROMPTS["critic"]}
    return build_deep_agent(
        models["coordinator"], subagents=[researcher, critic], system_prompt=PROMPTS["coordinator"],
        checkpointer=checkpointer, model_calls=12, tool_calls=20, fallbacks=False,
        permissions=[*secret_deny_rules(),
                     FilesystemPermission(operations=["write"], paths=["/reports/**"], mode="interrupt")])

---
## Step 4 · The review policy

A human approves every brief before it's saved. Write the policy as code: it is what you'd show the reviewer as a checklist, and what lets the evaluation run unattended.

In [ ]:
def review(action):
    """The human-review policy for a pending /reports/ write: approve or reject with reasons."""
    brief = action["args"].get("content", "")
    problems = [f"missing section {s}" for s in BRIEF_SECTIONS if not section(brief, s)]
    declined = "not found" in section(brief, "Answer").lower()
    sourced = bool(cited_docs(brief) or re.search(r"https?://", section(brief, "Sources")))
    if not declined and not sourced:
        problems.append("an answer without sources")
    if declined and "low" not in section(brief, "Confidence").lower():
        problems.append("a declined answer must have low confidence")
    return {"type": "reject", "message": "; ".join(problems)} if problems else {"type": "approve"}

In [ ]:
GOOD = brief_markdown("Q?", "US$349.", [{"source": "products/nimbus-pro.md", "quote": "Price: US$349"}], confidence="high")
NO_SOURCE = brief_markdown("Q?", "US$349.", [], confidence="high")
BAD_DECLINE = brief_markdown("Q?", "Not found in the documents or on the web.", [], confidence="high")
NO_SECTION = GOOD.replace("## Confidence", "## Notes")
DECLINE = brief_markdown("Q?", "Not found in the documents or on the web.", [], confidence="low: no source")
fixtures = {"good": (GOOD, "approve"), "no source": (NO_SOURCE, "reject"), "bad decline": (BAD_DECLINE, "reject"),
            "no section": (NO_SECTION, "reject"), "decline": (DECLINE, "approve")}
review_ok = {}
for name, (text, want) in fixtures.items():
    got = review({"name": "write_file", "args": {"file_path": "/reports/x.md", "content": text}})
    review_ok[name] = got["type"] == want
    print(f"{name:<12} -> {got['type']:<8} {'✅' if review_ok[name] else '❌ expected ' + want} {got.get('message', '')}")

**You should now see** five ✅: two approvals and three rejections with their reasons (solution). Starter: three ❌ (it approves everything).

---
## Step 5 · One question end to end, and the checker

In [ ]:
# 🧪 Offline: scripted models, free.
def run_question(q, models, tools, *, max_reviews=3):
    """One question end to end: invoke, answer every review interrupt with review(), resume."""
    ledger.clear()
    agent = build_analyst(models, tools, checkpointer=InMemorySaver())
    if agent is None:
        return None
    config = {"configurable": {"thread_id": q["id"]}}
    result = agent.invoke({"messages": [{"role": "user", "content": q["question"]}]}, config)
    reviews = []
    while "__interrupt__" in result and len(reviews) < max_reviews:
        actions = result["__interrupt__"][0].value["action_requests"]
        decisions = [review(a) for a in actions]
        reviews += decisions
        result = agent.invoke(Command(resume={"decisions": decisions}), config)
    briefs = [p for p in result.get("files", {}) if p.startswith("/reports/")]
    return {"id": q["id"], "result": result, "reviews": reviews, "sources": list(ledger.docs) + list(ledger.urls),
            "brief": file_text(result, briefs[0]) if briefs else None, "brief_path": briefs[0] if briefs else None}


demo = run_question(QUESTIONS[2], offline_models(), (search_docs, web_search))
if demo:
    print("trajectory:", trajectory(demo["result"]), "| subagents:", subagents_called(demo["result"]))
    print("reviews   :", demo["reviews"])
    print(demo["brief"])
else:
    print("❌ build_analyst returned None: do TODO 2.")

**You should now see** (with TODO 2) the trajectory `['task', 'task', 'write_file']`, the subagents `['researcher', 'critic']`, one `approve` and a brief with the four sections citing `[doc: support/faq-wifi.md]`.

In [ ]:
def check_brief(run, q):
    """Checks for one run (all bool). Uses only the run, the question and the ledger, never the agent's own claims."""
    brief = run["brief"] or ""
    answer = section(brief, "Answer").lower()
    cited = cited_docs(brief)
    path = trajectory(run["result"])
    order = [c for c in subagents_called(run["result"])]
    checks = {
        "brief": all(section(brief, s) for s in BRIEF_SECTIONS),
        "grounded": set(cited) <= set(run["sources"]),
        "budget": run_stats(run["result"])["model_calls"] <= 12,
        "workflow": order[:2] == ["researcher", "critic"] and path[-1:] == ["write_file"],
    }
    if q["in_corpus"]:
        checks["correct"] = q["answer_contains"].lower() in answer and bool(set(cited) & set(q["expected_sources"]))
    else:
        checks["correct"] = "not found" in answer and not cited
    checks["passed"] = all(checks.values())
    return checks

In [ ]:
def fake_run(brief, sources, subagents=("researcher", "critic")):
    from langchain_core.messages import AIMessage
    msgs = [AIMessage("", tool_calls=[{"name": "task", "args": {"subagent_type": s}, "id": s}]) for s in subagents]
    msgs.append(AIMessage("", tool_calls=[{"name": "write_file", "args": {}, "id": "w"}]))
    return {"brief": brief, "sources": sources, "result": {"messages": msgs + [AIMessage("done")]}}


q01 = QUESTIONS[0]
pro = "products/nimbus-pro.md"
good = brief_markdown(q01["question"], "It costs US$349.", [{"source": pro, "quote": "Price: US$349"}], confidence="high")
invented = brief_markdown(q01["question"], "It costs US$349.", [{"source": "products/nimbus-ultra.md", "quote": "x"}], confidence="high")
cases = {"good run": (fake_run(good, [pro]), True), "invented source": (fake_run(invented, [pro]), False),
         "critic skipped": (fake_run(good, [pro], ("researcher",)), False),
         "wrong answer": (fake_run(good.replace("US$349", "US$299"), [pro]), False)}
checker_ok = {}
for name, (run, want) in cases.items():
    got = check_brief(run, q01)
    checker_ok[name] = got.get("passed") is want and len(got) > 1
    print(f"{name:<16} passed={got.get('passed')!s:<5} {'✅' if checker_ok[name] else '❌'} {got}")

**You should now see** four ✅: the good run passes; the invented source, the skipped critic and the wrong price fail (solution).

---
## Step 6 · Evaluate on all 20 questions (offline)

In [ ]:
# 🧪 Offline: the scripted models on ALL 20 questions (free). LIVE runs the smoke set unless LIVE_FULL = True.
def evaluate(questions, models_factory, tools):
    rows = []
    for q in questions:
        run = run_question(q, models_factory(), tools)
        if run is None:
            return []
        rows.append({**run, "checks": check_brief(run, q), "stats": run_stats(run["result"])})
    return rows


rows = evaluate(QUESTIONS, offline_models, (search_docs, web_search))
passed = sum(r["checks"].get("passed", False) for r in rows)
interval = wilson_interval(passed, len(rows)) if rows else None
failing = {}
for r in rows:
    for name, ok in r["checks"].items():
        if not ok and name != "passed":
            failing.setdefault(name, []).append(r["id"])
print(f"offline: {passed}/{len(rows)} passed", f"{interval:.0%}" if interval else "")
print("failing checks:", failing)
declined = [r for r, q in zip(rows, QUESTIONS) if not q["in_corpus"] and r["checks"].get("correct")]
print("declined correctly:", len(declined), "of", sum(not q["in_corpus"] for q in QUESTIONS))

**You should now see** (solution) `17/20 passed` with an interval of about `[64%, 95%]`, all four out-of-scope questions declined, and the failures under `correct`: the offline reader quoted the wrong sentence for `g01`, `g02` and missed `g10`. That's the real baseline to beat with a model.

---
## Step 7 · 💸 Optional: the analyst on real models

> 💸 Calls your deep model (`MASTERY_DEEP_MODEL`, default Gemini free tier). Rate-limited to one call every 4 s and capped per run. Without the key it prints `⏭️ Skipped`.

In [ ]:
# 💸 LIVE (opt-in): the same analyst on your real models. Smoke (3 questions) ≈ 45–75K tokens; LIVE_FULL (20) ≈ 300–500K.
RUN_LIVE = False
LIVE_FULL = False      # 20 questions: spread over several days on a free tier
LIVE_WEB = False       # True: real web search (Groq browser search or Tavily, Lesson 05.01)
live_rows = []
if RUN_LIVE and HAVE_DEEP_KEY:
    from mastery.deep import deep_model
    live_tools = research_tools(ledger, live_web=LIVE_WEB)

    def live_models():
        return {"coordinator": deep_model("main"), "researcher": deep_model("subagent"), "critic": deep_model("subagent")}

    live_rows = evaluate(QUESTIONS if LIVE_FULL else SMOKE, live_models, live_tools)
    for r in live_rows:
        print(r["id"], "✅" if r["checks"]["passed"] else "❌", {k: v for k, v in r["checks"].items() if not v},
              "| calls", r["stats"]["model_calls"], "| tokens", r["stats"]["input_tokens"] + r["stats"]["output_tokens"])
    live_passed = sum(r["checks"]["passed"] for r in live_rows)
    print(f"LIVE: {live_passed}/{len(live_rows)} passed", f"{wilson_interval(live_passed, len(live_rows)):.0%}")
elif not RUN_LIVE:
    print("⏭️ Skipped: set RUN_LIVE = True (smoke set ≈ 45–75K tokens; coordinator token counts only, subagents add more).")
else:
    print(f"⏭️ Skipped: no key for {DEEP_MODEL} (see the setup cell).")

**You should now see** (when enabled) one line per question with the failing checks, calls and tokens, then the LIVE pass rate. Read every failing brief before changing a prompt.

---
## Step 8 · The report

In [ ]:
def write_report(path):
    ship = interval is not None and interval.low >= 0.6 and len(declined) == 4
    worst = sorted(failing.items(), key=lambda kv: -len(kv[1]))
    tokens = [r["stats"]["input_tokens"] + r["stats"]["output_tokens"] for r in rows]
    lines = [
        "# Project 11 report: Deep Research Analyst", "",
        "## Decision", f"**{'SHIP' if ship else 'HOLD'}** (bar: pass-rate lower bound ≥ 60% and every out-of-scope question declined).", "",
        "## Results", f"- Offline (scripted models): {passed}/{len(rows)} briefs passed, {interval:.1%}.",
        f"- Declined correctly: {len(declined)}/4.",
        f"- LIVE: {sum(r['checks']['passed'] for r in live_rows)}/{len(live_rows)} passed." if live_rows else "- LIVE: not run.", "",
        "## Failures", *([f"- `{name}`: {len(ids)} ({', '.join(ids)})" for name, ids in worst] or ["- none"]), "",
        "## Cost", f"- Model calls per run (coordinator): max {max(r['stats']['model_calls'] for r in rows)}.",
        f"- Tokens per run (coordinator, offline estimate): mean {sum(tokens) / len(tokens):.0f}.", "",
        "## Risks and next steps",
        "- The offline reader quotes the best sentence; a real model must be measured on the smoke set before use.",
        "- Free-tier prompts may be read by humans: synthetic/public documents only.",
        "- Next: fix the top failing check, then re-run the full set on two different days.",
    ]
    Path(path).write_text("\n".join(lines) + "\n", encoding="utf-8")
    return "\n".join(lines)

In [ ]:
REPORT_PATH = OUT_DIR / "project11-report.md"
report_text = write_report(REPORT_PATH)
print(report_text[:900])
if rows:      # also save one approved brief: the deliverable you would actually send
    best = next((r for r in rows if r["checks"].get("passed")), None)
    if best:
        (OUT_DIR / "project11-example-brief.md").write_text(best["brief"], encoding="utf-8")

**You should now see** the report beginning `# Project 11 report` and `**SHIP**` (solution). Saved to `data/outputs/m11/project11-report.md`, with one approved brief in `project11-example-brief.md`.

---
## Acceptance checks

In [ ]:
# 🧪 Offline: every check recomputes from the runs above (no model calls).
def sec(name):
    return section(report_text, name)


env_denied = False
if (probe := build_analyst(offline_models(), (search_docs, web_search), checkpointer=InMemorySaver())) is not None:
    from mastery.testing import ScriptedChatModel
    from langchain_core.messages import AIMessage
    bad = {"coordinator": ScriptedChatModel(responses=[AIMessage("", tool_calls=[{"name": "read_file", "args": {"file_path": "/.env"}, "id": "e"}]), AIMessage("x")]),
           "researcher": offline_models()["researcher"], "critic": offline_models()["critic"]}
    out = build_analyst(bad, (search_docs, web_search), checkpointer=InMemorySaver()).invoke(
        {"messages": [{"role": "user", "content": "read the env"}]}, {"configurable": {"thread_id": "env"}})
    env_denied = any("permission denied" in m.text for m in out["messages"] if m.type == "tool")
checks = {
    "1 the three prompts cover the workflow, sections and citation style": not prompt_problems(PROMPTS),
    "2 the analyst delegates to researcher then critic, and pauses before every /reports/ write":
        bool(rows) and all(r["reviews"] for r in rows) and all(r["checks"].get("workflow") for r in rows),
    "3 secrets are denied (/.env)": env_denied,
    "4 the review policy approves good briefs and rejects the three bad ones": all(review_ok.values()),
    "5 check_brief passes a good run and fails invented sources, a skipped critic and a wrong answer": all(checker_ok.values()),
    "6 offline: ≥ 80% of the 20 briefs pass and all 4 out-of-scope questions are declined":
        bool(rows) and passed / len(rows) >= 0.8 and len(declined) == 4,
    "7 every run stays within 12 model calls": bool(rows) and all(r["stats"]["model_calls"] <= 12 for r in rows),
    "8 the report has all sections, the measured numbers and a decision that follows the bar":
        all(sec(s) for s in ("Decision", "Results", "Failures", "Cost", "Risks and next steps"))
        and f"{passed}/{len(rows)}" in report_text and interval is not None and f"{interval:.1%}" in report_text
        and ("SHIP" in sec("Decision")) == (interval.low >= 0.6 and len(declined) == 4),
}
for name, ok in checks.items():
    print("✅" if ok else "❌", name)
print(f"\n{sum(checks.values())}/{len(checks)} acceptance checks passed")
_ = (OUT_DIR / "project11-acceptance.json").write_text(json.dumps(checks, indent=2), encoding="utf-8")

**You should now see** `8/8 acceptance checks passed` (solution). Starter: `0/8`. The checks can't be passed with shortcuts: the review policy and the checker are tested on fixtures with known answers, the secrets rule is probed with a real `read_file("/.env")`, the eval recomputes every brief, and the report's decision must follow the bar. The result is saved to `data/outputs/m11/project11-acceptance.json`.

## Findings (write yours)

- Which check failed most on real models, and was the cause the prompt, the tools or the documents?
- Did the critic ever change a brief? If not, is it worth its tokens?
- What would you need before sending these briefs to customers (paid tier, human review rate, monitoring)?

## ✅ Before you commit

- [ ] Restart & Run All passes without a key.
- [ ] All TODOs done; 8/8 acceptance checks; report and findings written.
- [ ] Outputs cleared (`jupyter nbconvert --clear-output --inplace <notebook>`); `data/outputs/` not committed.

---
## ➡️ What's next

🎉 **Module 11 done:** you built, secured, evaluated and reported on a deep agent the way a production team would.

**Next:** see [COURSE_OUTLINE.md](../../../../COURSE_OUTLINE.md) for the following module. Back to the [Module 11 README](../../../../modules/11-deep-agents/README.md).